# assignment 3: Use Cases for the Supervisor Pattern
A different use case for the supervisor pattern is a Research Paper Assistant. In this system, the user may ask the assistant to summarize several research papers, identify the common themes among them, prepare presentation points, and generate discussion questions. This is a good fit for the supervisor pattern because the task includes several different types of work, not just one.
The system can be divided into specialized subagents. A Paper Summary Agent focuses on summarizing individual papers. A Theme Extraction Agent compares the summaries and finds common ideas. A Presentation Agent turns the results into short bullet points for slides. A Discussion Question Agent creates useful questions for class discussion or presentation defense. Above these subagents, a Supervisor Agent decides which subagent to call and in what order, then combines all outputs into one final answer.

One agent would not be enough to accomplish this task alone because the problem requires different kinds of reasoning. Summarizing a paper is different from comparing several papers. Preparing presentation points is also different from generating discussion questions. If one agent had to handle everything at once, it would need to manage too many tools and responsibilities, which could reduce accuracy and organization. The supervisor pattern solves this problem by dividing the work among focused subagents while keeping one central supervisor in control of the overall workflow.

This architecture makes the system more modular, reliable, and easier to improve later. For example, a better summarization agent could be added without changing the presentation or question-generation agents. For these reasons, the Research Paper Assistant is a strong example of when the supervisor pattern is more effective than using a single agent.


In [2]:
# Assignment 3: Use Cases for the Supervisor Pattern
# Use Case: Research Paper Assistant with Subagents

"""
Overview
--------
A good use case for the supervisor pattern is a Research Paper Assistant.
This system helps a user with academic work such as:
- summarizing multiple research papers,
- identifying common themes across them,
- preparing presentation points,
- generating discussion questions.

Why use a supervisor?
---------------------
This task is not just one simple action. It combines several different types of work:
1. Reading and summarizing papers
2. Comparing papers to find shared themes
3. Turning results into presentation points
4. Generating discussion questions

One agent would not be enough because each of these tasks requires different reasoning,
different prompts, and sometimes different tools. A single agent with access to everything
could become less accurate and more confused. The supervisor pattern is better because
the supervisor coordinates specialized subagents, and each subagent focuses on one job only.

Architecture
------------
Bottom layer:
- Structured tools that perform specific actions

Middle layer:
- Specialized subagents:
    1. Paper Summary Agent
    2. Theme Extraction Agent
    3. Presentation Agent
    4. Discussion Question Agent

Top layer:
- Supervisor Agent
  Routes the user request to the correct subagents and combines the results
"""

# ---------------------------------------------------
# 1) Install and import dependencies
# ---------------------------------------------------

!pip install "langchain[openai]" langchain-openai -q

import os
from google.colab import userdata
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage

import os
from langchain_openai import ChatOpenAI

os.environ["OPENROUTER_API_KEY"] = "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"

model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
# ---------------------------------------------------
# 2) Define low-level tools
# ---------------------------------------------------

@tool
def summarize_paper(paper_text: str) -> str:
    """Summarize the main idea, method, and findings of a research paper."""
    return (
        "Paper Summary:\n"
        "- Main idea: The paper studies an AI-based solution for a real-world problem.\n"
        "- Method: It uses machine learning and data analysis techniques.\n"
        "- Findings: The results show improved performance compared to traditional methods."
    )

@tool
def extract_common_themes(paper_summaries: list[str]) -> str:
    """Extract common themes and repeated ideas from multiple paper summaries."""
    return (
        "Common Themes:\n"
        "1. Use of AI and machine learning for automation\n"
        "2. Importance of data quality and preprocessing\n"
        "3. Focus on improving accuracy and efficiency\n"
        "4. Need for real-world evaluation and practical deployment"
    )

@tool
def create_presentation_points(themes: str) -> str:
    """Turn extracted themes into short presentation bullet points."""
    return (
        "Presentation Points:\n"
        "- Many papers use AI to automate complex tasks\n"
        "- Data preprocessing is critical for good performance\n"
        "- Most studies aim to improve efficiency and prediction accuracy\n"
        "- Real-world testing is important before deployment"
    )

@tool
def generate_discussion_questions(themes: str) -> str:
    """Generate discussion questions based on the common themes."""
    return (
        "Discussion Questions:\n"
        "1. Why is data preprocessing important in AI systems?\n"
        "2. What challenges appear when AI models are used in real environments?\n"
        "3. How can researchers measure both accuracy and practical usefulness?\n"
        "4. What are the limitations of current AI-based approaches?\n"
        "5. How could these studies be improved in future work?"
    )

# ---------------------------------------------------
# 3) Create specialized sub-agents
# ---------------------------------------------------

PAPER_SUMMARY_AGENT_PROMPT = (
    "You are a research paper summarization assistant. "
    "Read natural language requests about academic papers. "
    "Use summarize_paper to create a clear summary of the paper's main idea, method, and findings. "
    "Always return a clean final summary."
)

THEME_AGENT_PROMPT = (
    "You are a theme extraction assistant. "
    "Analyze multiple paper summaries and identify their common themes. "
    "Use extract_common_themes when needed. "
    "Always return a clean final theme analysis."
)

PRESENTATION_AGENT_PROMPT = (
    "You are a presentation preparation assistant. "
    "Turn research findings and themes into concise presentation bullet points. "
    "Use create_presentation_points when needed. "
    "Always return presentation-ready points."
)

DISCUSSION_AGENT_PROMPT = (
    "You are a discussion question assistant. "
    "Generate useful academic discussion questions based on research themes. "
    "Use generate_discussion_questions when needed. "
    "Always return clear questions suitable for class discussion or presentation defense."
)

paper_summary_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[summarize_paper],
    system_prompt=PAPER_SUMMARY_AGENT_PROMPT,
)

theme_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[extract_common_themes],
    system_prompt=THEME_AGENT_PROMPT,
)

presentation_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[create_presentation_points],
    system_prompt=PRESENTATION_AGENT_PROMPT,
)

discussion_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[generate_discussion_questions],
    system_prompt=DISCUSSION_AGENT_PROMPT,
)

# ---------------------------------------------------
# 4) Wrap sub-agents as tools for the supervisor
# ---------------------------------------------------

@tool
def summarize_research_papers(request: str) -> str:
    """
    Summarize research papers using natural language.
    Use this when the user wants summaries of one or more academic papers.
    """
    result = paper_summary_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

@tool
def analyze_research_themes(request: str) -> str:
    """
    Identify common themes across multiple research papers.
    Use this when the user wants cross-paper comparison or theme extraction.
    """
    result = theme_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

@tool
def prepare_research_presentation(request: str) -> str:
    """
    Create presentation points from paper summaries or themes.
    Use this when the user wants presentation-ready bullets.
    """
    result = presentation_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

@tool
def prepare_discussion_questions(request: str) -> str:
    """
    Generate discussion or defense questions from research themes.
    Use this when the user wants questions for a class, seminar, or presentation.
    """
    result = discussion_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

# ---------------------------------------------------
# 5) Create the supervisor agent
# ---------------------------------------------------

SUPERVISOR_PROMPT = (
    "You are a helpful Research Paper Assistant supervisor. "
    "You coordinate specialized subagents to help users work with academic papers. "
    "You can summarize papers, identify common themes, prepare presentation points, "
    "and generate discussion questions. "
    "When the user asks for multiple actions, call multiple tools in sequence and "
    "then combine the results into one final response."
)

supervisor_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        summarize_research_papers,
        analyze_research_themes,
        prepare_research_presentation,
        prepare_discussion_questions,
    ],
    system_prompt=SUPERVISOR_PROMPT,
)

# ---------------------------------------------------
# 6) Test the supervisor with a multi-step request
# ---------------------------------------------------

query = """
I have several research papers about using artificial intelligence in healthcare.
Please summarize the papers, identify the common themes, prepare presentation points,
and generate discussion questions for my class presentation.

Paper 1: This paper uses machine learning to predict disease risk from patient records.
Paper 2: This paper applies deep learning to medical image classification.
Paper 3: This paper discusses challenges of deploying AI systems in hospitals.
"""

result = supervisor_agent.invoke({"messages": [HumanMessage(query)]})

for message in result.get("messages", []):
    message.pretty_print()

# ---------------------------------------------------
# 7) Justification
# ---------------------------------------------------

print("""
Justification:
This use case is a strong example of the supervisor pattern because it involves
multiple distinct tasks: summarization, comparison, presentation preparation,
and question generation. One agent would not be enough because these tasks require
different reasoning styles and different tools. A single agent handling everything
could become less accurate and less organized. By using the supervisor pattern,
the supervisor agent delegates each task to a specialized subagent and then combines
their outputs into one coherent final response. This makes the system more modular,
more reliable, and easier to improve in the future.
""")

================================ Human Message =================================


I have several research papers about using artificial intelligence in healthcare.
Please summarize the papers, identify the common themes, prepare presentation points,
and generate discussion questions for my class presentation.

Paper 1: This paper uses machine learning to predict disease risk from patient records.
Paper 2: This paper applies deep learning to medical image classification.
Paper 3: This paper discusses challenges of deploying AI systems in hospitals.

================================== Ai Message ==================================
Tool Calls:
  summarize_research_papers (call_6644761874a24cb086e0d20b)
 Call ID: call_6644761874a24cb086e0d20b
  Args:
    request: Summarize the following three research papers: Paper 1: This paper uses machine learning to predict disease risk from patient records. Paper 2: This paper applies deep learning to medical image classification. Paper 3: This paper 